# Housing Strand — Data Exploration and Training

**MultimodalAI'26 — Housing Demo**

This notebook covers data exploration, preprocessing, and model training for the housing strand demo.

**Structure**
1. Data Exploration — load, merge, and inspect the demo dataset.
2. Preprocessing & Split — feature engineering and property-level train/test split.
3. Model Training — train baseline model and save to disk.

**Prerequisite**
- `demo/data/raw/` CSV files must exist.

**Starter-kit boundary**
- This notebook trains and saves a model only.
- The Streamlit app (`app.py`) loads the saved model, evaluates it, and provides the submission form.

## Data Exploration

### Section 1 — Setup and Paths

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    DEMO_ROOT = cwd
elif (cwd / "demo" / "src").exists():
    DEMO_ROOT = cwd / "demo"
else:
    DEMO_ROOT = cwd.parent

sys.path.insert(0, str(DEMO_ROOT))
RAW_DIR = DEMO_ROOT / "data" / "raw"
PROCESSED_DIR = DEMO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("DEMO_ROOT:", DEMO_ROOT)
print("RAW_DIR:", RAW_DIR)

### Section 2 — Load and Merge Modalities

Load metadata + modality tables and merge into one daily table keyed by `reference`, `year`, `month`, `day`.

Creates the strand target label: `cold_risk = (avgTemperature < 19.0)`

In [ ]:
properties = pd.read_csv(RAW_DIR / "properties.csv")
env = pd.read_csv(RAW_DIR / "indoor_environment_daily.csv")
energy = pd.read_csv(RAW_DIR / "energy_noise_daily.csv")
survey = pd.read_csv(RAW_DIR / "resident_feedback_daily.csv")

key_cols = ["reference", "year", "month", "day"]
merged = env.merge(energy, on=key_cols, how="left").merge(survey, on=key_cols, how="left")
merged = merged.merge(properties, on="reference", how="left")

merged["date"] = pd.to_datetime(merged[["year", "month", "day"]], errors="coerce")
merged = merged.sort_values(["reference", "date"]).reset_index(drop=True)
merged["cold_risk"] = (merged["avgTemperature"] < 19.0).astype(int)

print(f"Merged shape: {merged.shape}")
print(f"Properties: {merged['reference'].nunique()}")
print(f"Cold-risk prevalence: {merged['cold_risk'].mean():.3f}")

### Section 3 — Data Quality Snapshot

Compute missingness across core features and visualise.

In [ ]:
missing = merged[["avgTemperature", "avgHumidity", "avgCo2", "smart_meter_kwh", "noise_db", "survey_score"]].isna().mean().sort_values(ascending=False)
display(missing.to_frame("missing_rate"))

plt.figure(figsize=(8, 3))
sns.barplot(x=missing.index, y=missing.values)
plt.xticks(rotation=30, ha="right")
plt.ylabel("missing rate")
plt.title("Demo subset missingness")
plt.tight_layout()
plt.show()

### Section 4 — Save Merged Dataset

Save the merged table for use in the preprocessing step below.

In [ ]:
merged.to_csv(PROCESSED_DIR / "merged_demo.csv", index=False)
print("Saved:", PROCESSED_DIR / "merged_demo.csv")

## Preprocessing & Split

### Section 1 — Feature Engineering and Property-Level Split

Apply derived features and split by property reference (not row-level) to prevent leakage.

In [ ]:
from src.data_pipeline import engineer_features, split_by_property

feat_df = engineer_features(merged)
train_df, test_df = split_by_property(feat_df, test_size=0.25, seed=42)

train_df.to_csv(PROCESSED_DIR / "train_features.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_features.csv", index=False)

overlap = set(train_df["reference"]) & set(test_df["reference"])
print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Train properties: {train_df['reference'].nunique()} | Test properties: {test_df['reference'].nunique()}")
print(f"Property overlap (must be 0): {len(overlap)}")

### Section 2 — Split Summary

In [ ]:
summary = {
    "train_cold_risk_rate": round(train_df["cold_risk"].mean(), 3),
    "test_cold_risk_rate": round(test_df["cold_risk"].mean(), 3),
    "train_co2_missing_rate": round(train_df["avgCo2"].isna().mean(), 3),
    "test_co2_missing_rate": round(test_df["avgCo2"].isna().mean(), 3),
}
summary

In [ ]:
split_df = pd.concat([
    train_df.assign(split="train"),
    test_df.assign(split="test"),
], ignore_index=True)

plt.figure(figsize=(7, 3))
sns.countplot(data=split_df, x="property_type", hue="split")
plt.xticks(rotation=20, ha="right")
plt.title("Property-type distribution by split")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
sns.barplot(data=split_df, x="split", y="cold_risk")
plt.title("Cold-risk rate by split")
plt.tight_layout()
plt.show()

## Model Training

### Section 1 — Train Baseline and Save Model

Train a single logistic regression baseline and save the model binary. Metrics are computed later by the app.

In [ ]:
import joblib
from models.model_a import build_model_a

features = [
    "avgTemperature",
    "avgHumidity",
    "co2_imputed",
    "co2_missing",
    "smart_meter_kwh",
    "noise_db",
    "lag_temp",
    "day_of_week",
    "is_flat",
]

X_train = train_df[features]
y_train = train_df["cold_risk"]
X_test = test_df[features]
y_test = test_df["cold_risk"]

model = build_model_a()
model.fit(X_train, y_train)

MODELS_DIR = DEMO_ROOT / "saved_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "model_a_logistic_baseline.joblib"
joblib.dump(model, model_path)

print(f"Model saved to: {model_path}")

### Section 2 — Quick Sanity Check

Verify the saved model loads and produces predictions.

In [ ]:
loaded = joblib.load(model_path)
y_prob = loaded.predict_proba(X_test)[:, 1]
print(f"Test predictions shape: {y_prob.shape}")
print(f"Mean predicted probability: {y_prob.mean():.4f}")
print("\nDone. Run 'streamlit run app.py' to evaluate and submit.")